# fcnd quickstart\n\nThis notebook demonstrates 1-FCND, weighted FCND, and MSFCND on small synthetic data.

In [ ]:
import numpy as np
from fcnd import FCND, MSFCND, IsoForestLearner, OneClassSVMLearner
from fcnd.synthetic import generate_wset, gen_data, gen_data_weighted, sigmoid_weights


In [ ]:
rng = np.random.default_rng(0)
W = generate_wset(size=20, dims=6, random_state=rng)
X_ref = gen_data(W, 60, a=1.0, random_state=rng)
X_test = np.vstack([gen_data(W, 45, a=1.0, random_state=rng), gen_data(W, 15, a=3.0, random_state=rng)])
alpha = 0.1
X_ref.shape, X_test.shape


## 1-FCND

In [ ]:
fcnd = FCND(IsoForestLearner(random_state=0, n_estimators=100), use_numba=True)
res = fcnd.detect(X_ref, X_test, alpha=alpha, method='ebh')
res.n_rejections, res.rejections[:10]


## Weighted FCND\nWeighted FCND uses density-ratio-style weights under weighted exchangeability.

In [ ]:
theta = np.array([0.4, 0.2, 0.1, 0.0, 0.0, 0.0])
X_ref_w = gen_data(W, 60, a=1.0, random_state=rng)
X_test_w = np.vstack([
    gen_data_weighted(W, 45, a=1.0, theta=theta, random_state=rng),
    gen_data_weighted(W, 15, a=3.0, theta=theta, random_state=rng),
])
w_ref = sigmoid_weights(X_ref_w, theta)
w_test = sigmoid_weights(X_test_w, theta)
weighted = FCND(IsoForestLearner(random_state=1, n_estimators=100), use_numba=True)
wres = weighted.detect(X_ref_w, X_test_w, alpha=alpha, method='ebh', weights_ref=w_ref, weights_test=w_test)
wres.n_rejections, wres.rejections[:10]


## MSFCND\nMSFCND scores a candidate library and performs block-wise model selection before conformal calibration.

In [ ]:
learners = {
    'if50': IsoForestLearner(random_state=0, n_estimators=50),
    'if100': IsoForestLearner(random_state=1, n_estimators=100),
    'svm_auto': OneClassSVMLearner(nu=0.05, kernel='rbf', gamma='auto'),
}
training_modes = {'if50': 'is', 'if100': 'is', 'svm_auto': 'loo'}
ms = MSFCND(learners, alpha=alpha, K=min(10, X_test.shape[0]), training_modes=training_modes, selector_top_k=1, use_numba=True)
ms_res = ms.detect(X_ref, X_test, method='ebh')
ms_res.n_rejections, ms_res.metadata['selected_models'][:5]


## EBHCC note\n`fcnd.calibration.EBHCC` implements the streaming conditional-calibration variant. Its fast shortcuts are only enabled for the built-in p-value auxiliary statistics and hybrid `hatR` denominator used by the paper's implementations.